## Process output of simulations

In [1]:
import os
import glob
import gzip
import math
import random
import pickle

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.colors import LogNorm
import shapely.wkt as wkt
from shapely.geometry import Point, LineString, box
from shapely.ops import nearest_points
import lxml.etree as ET
import tqdm
import wandb
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset, Subset
import torch_geometric
from torch_geometric.data import Data, Batch
from torch_geometric.transforms import LineGraph
import processing_io as pio
import re 
import os
import glob
import math
import pickle

import numpy as np
import pandas as pd
import geopandas as gpd
import torch
from collections import defaultdict

import processing_io as pio
from torch_geometric.transforms import LineGraph

from torch_geometric.data import Data, Batch
import shapely.wkt as wkt
from tqdm import tqdm
import fiona
import os

import alphashape
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
from shapely.geometry import Point
import random

districts = gpd.read_file("../../../../data/visualisation/districts_paris.geojson")

# Parameters to adapt
districts_of_policy_implementation = [5, 6, 7]
string_is_for_1pm = "pop_1pm"
path = "../../../../data/" +  string_is_for_1pm + "_simulations/"
comparison_subdir = pio.get_subdirs(path + string_is_for_1pm + "_cap_reduction_in_zone_2/")

string_district_of_interest = "_".join([str(d) for d in districts_of_policy_implementation])
basecase_subdir = pio.get_subdirs(path + string_is_for_1pm + "_basecase/")

result_path_basecase_mean = "results/" + string_is_for_1pm + "_basecase_mean_links.geojson"
result_path_comparison_mean = "results/gdf_" + string_is_for_1pm + "_policy_in_" + string_district_of_interest + ".geojson"
result_path_difference = "results/gdf_" + string_is_for_1pm + "_difference.geojson"
result_path_basecase_mode_stats = "results/" + string_is_for_1pm + "_basecase_mean_mode_stats.csv"
result_path_comparison_mode_stats = "results/" + string_is_for_1pm + "_policy_in_" + string_district_of_interest + "_mean_mode_stats.csv"
result_path_percentage_diff_mode_stats = "results/" + string_is_for_1pm + "_policy_in_" + string_district_of_interest + "_percentage_diff_mode_stats.csv"

compute_comparison_with_basecase = True


In [2]:
random_seed_2_df_basecase_output_links = pio.create_dic_seed_2_output_links(subdir=basecase_subdir)
random_seed_2_df_basecase_eqasim_trips = pio.create_dic_seed_2_eqasim_trips(subdir=basecase_subdir)
basecase_output_links_gdfs = list(random_seed_2_df_basecase_output_links.values())
gdf_basecase_mean = pio.compute_average_or_median_geodataframe(geodataframes=basecase_output_links_gdfs, column_name="vol_car", is_mean=True)
gdf_basecase_mean = gdf_basecase_mean.rename(columns={"osm:way:highway": "highway"})

random_seed_2_df_comparison_output_links = pio.create_dic_seed_2_output_links(subdir=comparison_subdir)
random_seed_2_df_comparison_eqasim_trips = pio.create_dic_seed_2_eqasim_trips(subdir=comparison_subdir)
comparison_output_links_gdfs = list(random_seed_2_df_comparison_output_links.values())
gdf_comparison_mean = pio.compute_average_or_median_geodataframe(geodataframes=comparison_output_links_gdfs, column_name="vol_car", is_mean=True)
gdf_comparison_mean = gdf_comparison_mean.rename(columns={"osm:way:highway": "highway"})

# For the basecase mean, we remove duplicate entries.

In [3]:
remaining_duplicates = pio.find_duplicate_edges_in_gdf(gdf_basecase_mean)
print(f"Number of duplicate edges: {len(remaining_duplicates)}")

links_without_duplicates = pio.summarize_duplicate_edges(gdf_basecase_mean)
print(f"Number of remaining duplicate edges after summarizing: {len(pio.find_duplicate_edges_in_gdf(links_without_duplicates))}")

Number of duplicate edges: 74
Number of remaining duplicate edges after summarizing: 0


In [4]:
detailed_entries, both_zero, both_nonzero, one_zero_one_nonzero = pio.identify_summarized_entries_detailed(gdf_basecase_mean, links_without_duplicates)

print(f"Number of summarized entries: {len(detailed_entries)}")
print(f"Entries where both 'vol_car' are zero: {len(both_zero)}")
print(f"Entries where both 'vol_car' are non-zero: {len(both_nonzero)}")
print(f"Entries where one 'vol_car' is zero and one is non-zero: {len(one_zero_one_nonzero)}")

# def print_examples(category, entries, num_examples=2):
#     print(f"\n{category} (showing {min(num_examples, len(entries))} examples):")
#     for i, entry in enumerate(entries[:num_examples]):
#         print(f"\nExample {i+1}:")
#         print("Summarized row:")
#         print(entry['summarized'])
#         print("\nOriginal rows:")
#         print(entry['original'])
#         print(f"Count: {entry['count']}")
#         print("-" * 50)

# print_examples("Both 'vol_car' are zero", both_zero)
# print_examples("Both 'vol_car' are non-zero", both_nonzero)
# print_examples("One 'vol_car' is zero and one is non-zero", one_zero_one_nonzero)

Number of summarized entries: 74
Entries where both 'vol_car' are zero: 25
Entries where both 'vol_car' are non-zero: 9
Entries where one 'vol_car' is zero and one is non-zero: 40


## Fill Nan Values

The values freespeed and highway, which we will use later, have nan values. We need to approximate it.

In [5]:
# Filter entries where 'highway' is NaN
highway_nan_entries = links_without_duplicates[links_without_duplicates['highway'].isna()]

# Get the distribution of the column 'modes' for these entries
modes_distribution = highway_nan_entries['modes'].value_counts()

# Print the distribution
print("Distribution of 'modes' where 'highway' is NaN:")
print(modes_distribution)

# Check if all 'modes' are "pt,rail,train"
all_modes_are_pt_rail_train = (highway_nan_entries['modes'] == "pt,rail,train").all()
print("\nIs it true that when 'highway' is NaN, 'modes' is always 'pt,rail,train'?")
print(all_modes_are_pt_rail_train)

# For those entries where "highway" is currently NaN, set "highway" to "pt"
links_without_duplicates.loc[links_without_duplicates['highway'].isna(), 'highway'] = 'pt'

Distribution of 'modes' where 'highway' is NaN:
pt,rail,train                            971
artificial,stopFacilityLink,subway       641
artificial,subway                        609
rail                                     347
artificial,stopFacilityLink,tram         112
artificial,tram                          109
artificial,bus                            53
pt,subway                                  9
artificial,funicular,stopFacilityLink      4
artificial,bus,stopFacilityLink            3
artificial,rail                            2
artificial,funicular                       2
Name: modes, dtype: int64

Is it true that when 'highway' is NaN, 'modes' is always 'pt,rail,train'?
False


In [23]:
# Calculate the average freespeed for each highway type
average_freespeed_by_highway = links_without_duplicates.groupby('highway')['freespeed'].mean()

if 'pt' in average_freespeed_by_highway.index:
    average_freespeed_by_highway['pt'] = 0

# Fill NaN freespeed values with the average freespeed of their respective highway type
for highway_type, avg_freespeed in average_freespeed_by_highway.items():
    print(f"Highway type: {highway_type}")
    print(f"Average freespeed: {avg_freespeed}")
    if highway_type == 'pt':
        avg_freespeed = 0
        mask = (links_without_duplicates['highway'] == highway_type)
    else:
        mask = (links_without_duplicates['freespeed'].isna()) & (links_without_duplicates['highway'] == highway_type)
    links_without_duplicates.loc[mask, 'freespeed'] = avg_freespeed

# Check if there are any remaining NaN values in freespeed
remaining_nan = links_without_duplicates['freespeed'].isna().sum()
print(f"\nRemaining NaN values in freespeed: {remaining_nan}")

Highway type: construction
Average freespeed: 13.194444444444445
Highway type: living_street
Average freespeed: 5.100940052396363
Highway type: motorway
Average freespeed: 22.290688575899843
Highway type: motorway_link
Average freespeed: 15.2689313517339
Highway type: pedestrian
Average freespeed: 8.955938697318008
Highway type: primary
Average freespeed: 10.505286174793174
Highway type: primary_link
Average freespeed: 9.945208970438328
Highway type: pt
Average freespeed: 0.0
Highway type: residential
Average freespeed: 8.161352865144334
Highway type: secondary
Average freespeed: 9.1430231943784
Highway type: secondary_link
Average freespeed: 9.245439469320067
Highway type: service
Average freespeed: 10.0664767331434
Highway type: tertiary
Average freespeed: 8.513650453028136
Highway type: tertiary_link
Average freespeed: 9.146341463414634
Highway type: trunk
Average freespeed: 19.382371198013654
Highway type: trunk_link
Average freespeed: 12.022249522249522
Highway type: unclassified


In [24]:
gdf_to_save = links_without_duplicates.copy()
columns_to_drop = [col for col in gdf_to_save.columns if col.startswith('osm:')]
columns_to_drop.append('original_directions')
gdf_to_save.drop(columns=columns_to_drop, inplace=True)
gdf_to_save = gdf_to_save.set_crs("EPSG:4326", allow_override=True)

In [19]:
# Check the number of NaN values in freespeed
print(f"Number of NaN values in freespeed: {gdf_to_save['freespeed'].isna().sum()}")

Number of NaN values in freespeed: 0


In [22]:
gdf_to_save.to_file(result_path_basecase_mean, driver='GeoJSON')

In [10]:
gdf_comparison_mean_to_save = gdf_to_save[['link', 'geometry']].merge(gdf_comparison_mean, on='link', how='left')

gdf_comparison_mean_to_save.drop(columns=['geometry_x'], inplace=True)
gdf_comparison_mean_to_save.rename(columns={'geometry_y': 'geometry'}, inplace=True)
gdf_comparison_mean_to_save = gpd.GeoDataFrame(gdf_comparison_mean_to_save, geometry='geometry')
gdf_comparison_mean_to_save.crs = gdf_to_save.crs

# Save the dataframe
gdf_comparison_mean_to_save.to_file(result_path_comparison_mean, driver='GeoJSON')

In [11]:
def calculate_avg_mode_stats(single_mode_stats_list:list):
    mode_stats_list = []
    for df in single_mode_stats_list:
        mode_stats = df.groupby('mode').agg({
            'travel_time': ['mean', 'count'],
            'routed_distance': 'mean'
        }).reset_index()
        mode_stats.columns = ['mode', 'avg_travel_time', 'trip_count', 'avg_routed_distance']
        mode_stats_list.append(mode_stats)
    all_mode_stats = pd.concat(mode_stats_list, ignore_index=True)

    # Calculate the average across all seeds
    average_mode_stats = all_mode_stats.groupby('mode').agg({
        'avg_travel_time': 'mean',
        'avg_routed_distance': 'mean',
        'trip_count': 'mean'
    }).reset_index()
    average_mode_stats.columns = ['mode', 'avg_total_travel_time', 'avg_total_routed_distance', 'avg_trip_count']
    df_average_mode_stats = pd.DataFrame(average_mode_stats)
    return df_average_mode_stats

df_basecase_mode_stats = calculate_avg_mode_stats(random_seed_2_df_basecase_eqasim_trips.values())
df_basecase_mode_stats.to_csv(result_path_basecase_mode_stats, index=False)

df_comparison_mode_stats = calculate_avg_mode_stats(random_seed_2_df_comparison_eqasim_trips.values())
df_comparison_mode_stats.to_csv(result_path_comparison_mode_stats, index=False)

In [12]:
df_basecase_mode_stats

,mode,avg_total_travel_time,avg_total_routed_distance,avg_trip_count
0,bike,1187.134382,3681.674165,390.803922
1,car,941.046459,4835.211251,3750.470588
2,car_passenger,423.112003,4378.124317,938.000000
3,outside,0.788800,1057.635258,2756.235294
4,pt,1602.507719,5467.415175,4067.921569
5,walk,1007.712862,1209.811859,3232.000000


In [13]:
df_comparison_mode_stats

,mode,avg_total_travel_time,avg_total_routed_distance,avg_trip_count
0,bike,1187.799314,3683.735994,388.705882
1,car,1019.709389,4840.653576,3749.725490
2,car_passenger,423.747648,4381.807905,938.000000
3,outside,0.788843,1057.679029,2755.960784
4,pt,1603.134430,5464.151239,4070.568627
5,walk,1008.089650,1210.263787,3230.156863


In [14]:
# def calculate_avg_mode_stats(single_mode_stats_list:list):
#     mode_stats_list = []


#     for df in single_mode_stats_list:
#         mode_stats = df.groupby('mode').agg({
#             'travel_time': 'mean',
#             'routed_distance': 'mean'
#         }).reset_index()
#         mode_stats.columns = ['mode', 'avg_travel_time', 'avg_routed_distance']
#         mode_stats_list.append(mode_stats)
        
#     # Concatenate all mode_stats dataframes
#     all_mode_stats = pd.concat(mode_stats_list, ignore_index=True)

#     # Calculate the average across all seeds
#     average_mode_stats = all_mode_stats.groupby('mode').agg({
#         'avg_travel_time': 'mean',
#         'avg_routed_distance': 'mean'
#     }).reset_index()
#     average_mode_stats.columns = ['mode', 'avg_total_travel_time', 'avg_total_routed_distance']
#     df_average_mode_stats = pd.DataFrame(average_mode_stats)
#     return df_average_mode_stats

# df_basecase_mode_stats = calculate_avg_mode_stats(random_seed_2_df_basecase_eqasim_trips.values())
# df_basecase_mode_stats.to_csv(result_path_basecase_mode_stats, index=False)

# df_comparison_mode_stats = calculate_avg_mode_stats(random_seed_2_df_comparison_eqasim_trips.values())
# df_comparison_mode_stats.to_csv(result_path_comparison_mode_stats, index=False)

In [15]:
df_comparison_mode_stats

,mode,avg_total_travel_time,avg_total_routed_distance,avg_trip_count
0,bike,1187.799314,3683.735994,388.705882
1,car,1019.709389,4840.653576,3749.725490
2,car_passenger,423.747648,4381.807905,938.000000
3,outside,0.788843,1057.679029,2755.960784
4,pt,1603.134430,5464.151239,4070.568627
5,walk,1008.089650,1210.263787,3230.156863


In [16]:
df_basecase_mode_stats

,mode,avg_total_travel_time,avg_total_routed_distance,avg_trip_count
0,bike,1187.134382,3681.674165,390.803922
1,car,941.046459,4835.211251,3750.470588
2,car_passenger,423.112003,4378.124317,938.000000
3,outside,0.788800,1057.635258,2756.235294
4,pt,1602.507719,5467.415175,4067.921569
5,walk,1007.712862,1209.811859,3232.000000


In [17]:
# Compute the difference between the two dataframes
df_diff_mode_stats = df_comparison_mode_stats.set_index('mode') - df_basecase_mode_stats.set_index('mode')
df_diff_mode_stats.reset_index(inplace=True)

df_diff_mode_stats

# df_diff_mode_stats.to_csv(result_pa, index=False)

,mode,avg_total_travel_time,avg_total_routed_distance,avg_trip_count
0,bike,0.664932,2.061829,-2.098039
1,car,78.662930,5.442324,-0.745098
2,car_passenger,0.635645,3.683589,0.000000
3,outside,0.000043,0.043771,-0.274510
4,pt,0.626711,-3.263936,2.647059
5,walk,0.376787,0.451928,-1.843137


In [18]:
# Compute the percentage difference between the two dataframes
df_percentage_diff_mode_stats = df_diff_mode_stats.copy()
df_percentage_diff_mode_stats['avg_total_travel_time'] = (df_diff_mode_stats['avg_total_travel_time'] / df_basecase_mode_stats['avg_total_travel_time']) * 100
df_percentage_diff_mode_stats['avg_total_routed_distance'] = (df_diff_mode_stats['avg_total_routed_distance'] / df_basecase_mode_stats['avg_total_routed_distance']) * 100
df_percentage_diff_mode_stats['avg_trip_count'] = (df_diff_mode_stats['avg_trip_count'] / df_basecase_mode_stats['avg_trip_count']) * 100

pd.set_option('display.float_format', lambda x: '%.10f' % x)

df_percentage_diff_mode_stats

# Save the percentage difference dataframe to a CSV file
df_percentage_diff_mode_stats.to_csv(result_path_percentage_diff_mode_stats, index=False)

df_percentage_diff_mode_stats

,mode,avg_total_travel_time,avg_total_routed_distance,avg_trip_count
0,bike,0.0560115321,0.0560024773,-0.5368521399
1,car,8.3590910484,0.1125560845,-0.0198667880
2,car_passenger,0.1502309786,0.0841362255,0.0000000000
3,outside,0.0054561841,0.0041386152,-0.0099595925
4,pt,0.0391081188,-0.0596979797,0.0650715305
5,walk,0.0373903328,0.0373552435,-0.0570277616


In [19]:
# if compute_comparison_with_basecase:
#     random_seed_2_df_comparison = pio.create_dic_seed_2_output_links(subdir = comparison_subdir)
#     geodataframes_comparison = list(random_seed_2_df_comparison.values())
#     gdf_comparison_mean = pio.compute_average_or_median_geodataframe(geodataframes=geodataframes_comparison, column_name="vol_car", is_mean=True)
#     gdf_comparison_mean_extended = pio.extend_geodataframe(gdf_base = gdf_basecase_mean, gdf_to_extend=gdf_comparison_mean, column_to_extend='highway', new_column_name='highway')
#     gdf_basecase_without_unnecessary_columns = pio.remove_columns(gdf_with_correct_columns=gdf_comparison_mean_extended, gdf_to_be_adapted=gdf_basecase_mean)
#     gdf_basecase_difference = pio.compute_difference_geodataframe(gdf_to_substract_from=gdf_comparison_mean_extended, gdf_to_substract=gdf_basecase_without_unnecessary_columns, column_name= 'vol_car')
#     gdf_comparison_mean_extended.to_file(result_path_comparison_mean, driver='GeoJSON')
#     gdf_basecase_difference.to_file(result_path_difference, driver='GeoJSON')